In [1]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
from nltk.sentiment import SentimentIntensityAnalyzer
import speech_recognition as sr
from pydub import AudioSegment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import nltk
from wordcloud import WordCloud
from collections import Counter
import re

c:\Users\Jonny Villareal\AppData\Local\Programs\Python\Python312\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [2]:
irp = pd.read_excel('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/IRP/IRP_consolidado.xlsx')

irp.head()

,Fecha,NÃºmero FMS Bus,Servicio Bus,Id LÃ­nea,LÃ­nea,Ruta,Id Ruta,Tabla,Conductor,Id Viaje,...,dif_menor_10,Patio,Validaciones,Cantidad_frenado_brusco,Cantidad_exceso_velocidad,limite_velocidad,exceso_velocidad,Año,Mes,Semana del año
0,2026-04-24,502097,CE1360028,10688,BD237,BD237_V2,12758,28,508641,2,...,0,PATIO TINTAL 2,82.0,58.0,NaN,NaN,NaN,2026,4,17
1,2026-04-24,504143,CE1690029,10194,539,539_Vuelta_V2,12786,29,504185,2,...,0,PATIO TINTAL 2,113.0,10.0,NaN,NaN,NaN,2026,4,17
2,2026-04-24,504177,CE1690020,10194,539,539_Vuelta_V2,12786,22,507099,2,...,0,PATIO TINTAL 2,65.0,106.0,NaN,NaN,NaN,2026,4,17
3,2026-04-24,504418,CE1690024,10194,539,539_Vuelta_V2,12786,26,507583,2,...,0,PATIO TINTAL 2,132.0,33.0,NaN,NaN,NaN,2026,4,17
4,2026-04-24,507004,CE13F0006,10261,466,466_V2,12759,6,507369,2,...,0,PATIO TINTAL 2,42.0,54.0,NaN,NaN,NaN,2026,4,17


In [3]:
resumen = (
    irp
    .groupby(['Fecha', 'Conductor', 'Patio'], as_index=False)
    .agg({
        'cant_adelanto': 'sum',
        'adelanto_10_20': 'sum',
        'adelanto_mayor_20': 'sum',
        'dif_10_20': 'mean',
        'dif_mayor_20': 'mean',
        'Cantidad_frenado_brusco':'mean',
        'Cantidad_exceso_velocidad':'mean'
    })
)

resumen

,Fecha,Conductor,Patio,cant_adelanto,adelanto_10_20,adelanto_mayor_20,dif_10_20,dif_mayor_20,Cantidad_frenado_brusco,Cantidad_exceso_velocidad
0,2026-04-01,500221,PATIO TINTAL 2,254,63,3,6.625000,8.5,22.25,31.25
1,2026-04-01,500468,PATIO TINTAL 2,135,46,57,8.000000,11.5,116.00,NaN
2,2026-04-01,500716,PATIO TINTAL,150,28,69,14.500000,32.0,12.00,NaN
3,2026-04-01,500855,PATIO LA Y,157,86,24,13.666667,38.0,19.00,NaN
4,2026-04-01,500914,PATIO TINTAL 2,41,0,0,0.000000,0.0,25.00,7.50
...,...,...,...,...,...,...,...,...,...,...
28786,2026-05-03,510518,LA VERBENA_GM,78,0,0,0.000000,0.0,NaN,NaN
28787,2026-05-03,510520,LA VERBENA_GM,44,0,0,0.000000,0.0,NaN,NaN
28788,2026-05-03,510524,PATIO TINTAL,52,18,0,6.500000,0.0,NaN,NaN
28789,2026-05-03,510525,LA VERBENA_GM,7,0,0,0.000000,0.0,NaN,NaN


In [4]:
resumen['score'] = (
    resumen['cant_adelanto'] * 1 +
    resumen['adelanto_10_20'] * 2 +
    resumen['adelanto_mayor_20'] * 3 +
    resumen['dif_10_20'] * 0.5 +
    resumen['dif_mayor_20'] * 1 +
    resumen['Cantidad_frenado_brusco'] * 0.5 +
    resumen['Cantidad_exceso_velocidad'] * 1 
)

resumen

,Fecha,Conductor,Patio,cant_adelanto,adelanto_10_20,adelanto_mayor_20,dif_10_20,dif_mayor_20,Cantidad_frenado_brusco,Cantidad_exceso_velocidad,score
0,2026-04-01,500221,PATIO TINTAL 2,254,63,3,6.625000,8.5,22.25,31.25,443.1875
1,2026-04-01,500468,PATIO TINTAL 2,135,46,57,8.000000,11.5,116.00,NaN,NaN
2,2026-04-01,500716,PATIO TINTAL,150,28,69,14.500000,32.0,12.00,NaN,NaN
3,2026-04-01,500855,PATIO LA Y,157,86,24,13.666667,38.0,19.00,NaN,NaN
4,2026-04-01,500914,PATIO TINTAL 2,41,0,0,0.000000,0.0,25.00,7.50,61.0000
...,...,...,...,...,...,...,...,...,...,...,...
28786,2026-05-03,510518,LA VERBENA_GM,78,0,0,0.000000,0.0,NaN,NaN,NaN
28787,2026-05-03,510520,LA VERBENA_GM,44,0,0,0.000000,0.0,NaN,NaN,NaN
28788,2026-05-03,510524,PATIO TINTAL,52,18,0,6.500000,0.0,NaN,NaN,NaN
28789,2026-05-03,510525,LA VERBENA_GM,7,0,0,0.000000,0.0,NaN,NaN,NaN


In [5]:
resumen = resumen.sort_values(
    ['Fecha', 'Patio', 'score'],
    ascending=[True, True, False]
)

resumen['rank'] = (
    resumen
    .groupby(['Fecha', 'Patio'])
    .cumcount() + 1
)

resumen

,Fecha,Conductor,Patio,cant_adelanto,adelanto_10_20,adelanto_mayor_20,dif_10_20,dif_mayor_20,Cantidad_frenado_brusco,Cantidad_exceso_velocidad,score,rank
66,2026-04-01,503647,HIBRIDOS,107,63,26,14.00,11.500000,50.0,NaN,NaN,1
678,2026-04-01,509706,LA VERBENA_GM,313,96,127,11.40,14.600000,57.6,1.0,936.100000,1
324,2026-04-01,508085,LA VERBENA_GM,263,94,110,10.50,22.500000,46.5,20.0,852.000000,2
840,2026-04-01,510141,LA VERBENA_GM,227,27,167,9.00,22.666667,42.0,1.0,831.166667,3
698,2026-04-01,509759,LA VERBENA_GM,289,75,111,10.25,21.250000,31.0,1.0,814.875000,4
...,...,...,...,...,...,...,...,...,...,...,...,...
28485,2026-05-03,508836,PATIO TINTAL 2,1,0,0,0.00,0.000000,NaN,NaN,NaN,105
28490,2026-05-03,508879,PATIO TINTAL 2,144,12,0,2.75,0.000000,NaN,NaN,NaN,106
28494,2026-05-03,508936,PATIO TINTAL 2,58,5,0,2.50,0.000000,NaN,NaN,NaN,107
28500,2026-05-03,509008,PATIO TINTAL 2,30,2,0,2.50,0.000000,NaN,NaN,NaN,108


In [6]:
top20 = resumen[resumen['rank'] <= 20]

top20

,Fecha,Conductor,Patio,cant_adelanto,adelanto_10_20,adelanto_mayor_20,dif_10_20,dif_mayor_20,Cantidad_frenado_brusco,Cantidad_exceso_velocidad,score,rank
66,2026-04-01,503647,HIBRIDOS,107,63,26,14.000000,11.500000,50.0,NaN,NaN,1
678,2026-04-01,509706,LA VERBENA_GM,313,96,127,11.400000,14.600000,57.6,1.0,936.100000,1
324,2026-04-01,508085,LA VERBENA_GM,263,94,110,10.500000,22.500000,46.5,20.0,852.000000,2
840,2026-04-01,510141,LA VERBENA_GM,227,27,167,9.000000,22.666667,42.0,1.0,831.166667,3
698,2026-04-01,509759,LA VERBENA_GM,289,75,111,10.250000,21.250000,31.0,1.0,814.875000,4
...,...,...,...,...,...,...,...,...,...,...,...,...
28327,2026-05-03,504960,PATIO TINTAL 2,167,18,0,8.250000,0.000000,NaN,NaN,NaN,16
28329,2026-05-03,505210,PATIO TINTAL 2,58,29,12,6.000000,15.666667,NaN,NaN,NaN,17
28330,2026-05-03,505226,PATIO TINTAL 2,106,4,0,3.333333,0.000000,NaN,NaN,NaN,18
28333,2026-05-03,505606,PATIO TINTAL 2,263,53,55,9.500000,11.750000,NaN,NaN,NaN,19


In [7]:
resultado_final = irp.merge(
    top20[['Fecha', 'Conductor', 'Patio', 'score', 'rank']],
    on=['Fecha', 'Conductor', 'Patio'],
    how='inner'
)

resultado_final.head(2)

,Fecha,NÃºmero FMS Bus,Servicio Bus,Id LÃ­nea,LÃ­nea,Ruta,Id Ruta,Tabla,Conductor,Id Viaje,...,Validaciones,Cantidad_frenado_brusco,Cantidad_exceso_velocidad,limite_velocidad,exceso_velocidad,Año,Mes,Semana del año,score,rank
0,2026-04-24,507026,BC29DG034,10350,SE14,SE14_Ida_V3,12738,51,507187,2,...,47.0,34.0,NaN,NaN,NaN,2026,4,17,393.416667,16
1,2026-04-24,504316,CE12D0035,10264,614,614_Vuelta_V2,12328,34,509729,2,...,114.0,80.0,NaN,NaN,NaN,2026,4,17,157.100000,16


In [8]:
resultado_final.columns = resultado_final.columns.str.replace('Ã¡', 'a') \
                                                   .str.replace('Ã©', 'e') \
                                                   .str.replace('Ã­', 'i') \
                                                   .str.replace('Ã³', 'o') \
                                                   .str.replace('Ãº', 'u') \
                                                   .str.replace('Ã±', 'ñ')
                                                   
                                                   
resultado_final.head()

,Fecha,Numero FMS Bus,Servicio Bus,Id Linea,Linea,Ruta,Id Ruta,Tabla,Conductor,Id Viaje,...,Validaciones,Cantidad_frenado_brusco,Cantidad_exceso_velocidad,limite_velocidad,exceso_velocidad,Año,Mes,Semana del año,score,rank
0,2026-04-24,507026,BC29DG034,10350,SE14,SE14_Ida_V3,12738,51,507187,2,...,47.0,34.0,NaN,NaN,NaN,2026,4,17,393.416667,16
1,2026-04-24,504316,CE12D0035,10264,614,614_Vuelta_V2,12328,34,509729,2,...,114.0,80.0,NaN,NaN,NaN,2026,4,17,157.100000,16
2,2026-04-24,504423,CE1660018,10304,806,806_Vuelta_V3,12780,18,507036,2,...,47.0,32.0,NaN,NaN,NaN,2026,4,17,167.500000,15
3,2026-04-24,504423,CE1660018,10304,1902-03-16 00:00:00,806_Vuelta_V3,12780,18,507036,2,...,47.0,32.0,NaN,NaN,NaN,2026,4,17,167.500000,15
4,2026-04-21,502142,CE1630023,10310,C101,C101_Ida_V2,12777,23,502975,2,...,54.0,54.0,NaN,NaN,NaN,2026,4,17,NaN,9


In [9]:
columnas = [
    'Fecha',
    'Numero FMS Bus',
    'Servicio Bus',
    'Id Linea',
    'Linea',
    'Ruta',
    'Id Ruta',
    'Tabla',
    'Conductor',
    'Id Viaje',
    'Nombre de Conductor',
    'reg_ccz',
    'Supervisor',
    'cant_adelanto',
    'dif_adelanto',
    'tp26_neg',
    'adelanto_10_20',
    'adelanto_mayor_20',
    'dif_10_20',
    'dif_mayor_20',
    'Patio',
    'Validaciones',
    'Cantidad_frenado_brusco',
    'Cantidad_exceso_velocidad',	
    'rank'
    
]

resultado_final = resultado_final[columnas].copy()

resultado_final.head()

,Fecha,Numero FMS Bus,Servicio Bus,Id Linea,Linea,Ruta,Id Ruta,Tabla,Conductor,Id Viaje,...,tp26_neg,adelanto_10_20,adelanto_mayor_20,dif_10_20,dif_mayor_20,Patio,Validaciones,Cantidad_frenado_brusco,Cantidad_exceso_velocidad,rank
0,2026-04-24,507026,BC29DG034,10350,SE14,SE14_Ida_V3,12738,51,507187,2,...,5,0,0,0,0,PATIO TINTAL 2,47.0,34.0,NaN,16
1,2026-04-24,504316,CE12D0035,10264,614,614_Vuelta_V2,12328,34,509729,2,...,4,0,0,0,0,PATIO TINTAL,114.0,80.0,NaN,16
2,2026-04-24,504423,CE1660018,10304,806,806_Vuelta_V3,12780,18,507036,2,...,33,0,0,0,0,PATIO TINTAL,47.0,32.0,NaN,15
3,2026-04-24,504423,CE1660018,10304,1902-03-16 00:00:00,806_Vuelta_V3,12780,18,507036,2,...,34,0,0,0,0,PATIO TINTAL,47.0,32.0,NaN,15
4,2026-04-21,502142,CE1630023,10310,C101,C101_Ida_V2,12777,23,502975,2,...,4,0,0,0,0,LA VERBENA_GM,54.0,54.0,NaN,9


In [10]:
resultado_final.to_csv('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/IRP/top_adelantos.csv', index=False, sep=';')